# Progressive Disclosure on SWE-bench Pro

Demo of the `BaseUser` abstraction for multi-round agent runs on SWE-bench Pro.

**Tasks:** ansible, flipt, openlibrary, navidrome, qutebrowser (5 oracle-passing tasks).
**Baseline agent:** Gemini 3.1 Pro Preview, single-round.
**Progressive:** `BaseUser` callback, up to 3 rounds, hints disclosed on test failure.

Built for the SWE-bench Pro progressive-disclosure use case. Parity answer to [Harbor #1316](https://github.com/harbor-ai/harbor/issues/1316) for the no-second-LLM case — see [`docs/progressive-disclosure.md`](../docs/progressive-disclosure.md).

## Setup history (2026-04-24)

| Issue | Fix |
|-------|-----|
| `--rootdir` removed → test IDs came out as `../dev/::test_name`, verifier saw 0 passes despite all tests passing | Restored `--rootdir=/app` in `PYTEST_ADDOPTS` |
| `conftest.py` cleanup deleted qutebrowser's legitimate import-order conftests, breaking pytest collection (circular import) | Per-task `[verifier.hardening]` opt-out in `task.toml`: `cleanup_conftests = false` |

Result: oracle 2/4 → 5/5.

## 1. The `BaseUser` API

In [1]:
import inspect
import os

os.chdir('/workspace/repos/benchflow')

from benchflow import BaseUser, FunctionUser, RoundResult
from benchflow.trial import TrialConfig

print('BaseUser:', inspect.signature(BaseUser.run))
print('RoundResult fields:', list(RoundResult.__dataclass_fields__))
print('TrialConfig user fields:', [f for f in TrialConfig.__dataclass_fields__
                                    if f in ('user', 'max_user_rounds', 'oracle_access')])

BaseUser: (self, round: 'int', instruction: 'str', round_result: 'RoundResult | None' = None) -> 'str | None'
RoundResult fields: ['round', 'trajectory', 'rewards', 'verifier_output', 'verifier_error', 'n_tool_calls']
TrialConfig user fields: ['user', 'max_user_rounds', 'oracle_access']


## 2. A minimal progressive-disclosure user

Three-round progressive disclosure: terse prompt → hints with failing tests → full spec.

In [2]:
def progressive(round, instruction, rr):
    if round == 0:
        first_line = instruction.split('\n', 1)[0].strip()
        return f'{first_line}\n\nRead /app/ to understand the codebase, then implement the fix. Run tests when you think you are done.'

    if rr and rr.rewards:
        score = rr.rewards.get('reward', rr.rewards.get('exact_match', 0))
        if score >= 1.0:
            return None  # passed, stop

    if round == 1:
        half = len(instruction) // 2
        return (
            f'Tests failed:\n{(rr.verifier_output or "")[:1500]}\n\n'
            f'First half of spec:\n{instruction[:half]}'
        )

    if round == 2:
        return f'Full spec:\n{instruction}'

    return None

user = FunctionUser(progressive)
print(f'User type: {type(user).__name__}')

User type: FunctionUser


## 3. Oracle validation: 5/5 SWE-bench Pro tasks pass

Verifies the gold solution (`solve.sh`) reaches reward=1.0 on every task. Required before trusting baseline/progressive numbers — without a working oracle we can't tell if low rewards mean agent failure or verifier failure.

In [3]:
import csv
from pathlib import Path

results = list(csv.DictReader(open('experiments/swebench-pro-results.csv')))

# qutebrowser oracle in CSV is from BEFORE the hardening opt-out fix.
# Post-fix oracle: 1.0 (verified separately on 2026-04-24).
qutebrowser_post_fix = {'task': 'qutebrowser', 'experiment': 'oracle',
                         'reward': '1.0', 'elapsed_s': '24.5',
                         'note': 'after [verifier.hardening] cleanup_conftests=false'}
navidrome_oracle = {'task': 'navidrome', 'experiment': 'oracle',
                     'reward': '1.0', 'elapsed_s': '~120',
                     'note': 'verified separately'}

oracle = [r for r in results if r['experiment'] == 'oracle' and r['task'] != 'qutebrowser']
for r in oracle:
    r['note'] = ''
oracle.append(qutebrowser_post_fix)
oracle.append(navidrome_oracle)

print(f'{"Task":<14} {"Reward":<8} {"Time(s)":<8} Notes')
print('-' * 60)
for r in oracle:
    print(f'{r["task"]:<14} {r["reward"]:<8} {r["elapsed_s"]:<8} {r["note"]}')

Task           Reward   Time(s)  Notes
------------------------------------------------------------
ansible        1.0      54.3     
flipt          1.0      480.5    
openlibrary    1.0      147.7    
qutebrowser    1.0      24.5     after [verifier.hardening] cleanup_conftests=false
navidrome      1.0      ~120     verified separately


## 4. Baseline: Gemini 3.1 Pro single-round

Each task gets the full SWE-bench Pro instruction in one shot. No user, no rounds.

In [4]:
baseline = [r for r in results if r['experiment'] == 'baseline']

print(f'{"Task":<14} {"Reward":<8} {"Tools":<6} {"Time(s)":<8}')
print('-' * 42)
for r in baseline:
    print(f'{r["task"]:<14} {r["reward"]:<8} {r["n_tool_calls"]:<6} {r["elapsed_s"]:<8}')

passed = sum(1 for r in baseline if float(r['reward']) >= 1.0)
print(f'\nBaseline (Gemini 3.1 Pro, 1 round): {passed}/{len(baseline)} passed')

Task           Reward   Tools  Time(s) 
------------------------------------------
ansible        1.0      23     206.8   
flipt          0.0      61     1443.7  
openlibrary    1.0      32     340.0   
qutebrowser    0.0      72     543.6   

Baseline (Gemini 3.1 Pro, 1 round): 2/4 passed


## 5. Progressive disclosure: 5-task run with Gemini 3.1 Pro

Three-round progressive disclosure (terse → failing tests + half spec → full spec) on all 5 oracle-passing SWE-bench Pro tasks. Daytona backend, Gemini 3.1 Pro Preview as the agent.

**What we expected:**
- **flipt** — baseline failed at 24min/61 tools; progressive should give the agent failure feedback to course-correct
- **openlibrary** — baseline already passed; progressive should not regress
- **ansible / navidrome / qutebrowser** — see whether progressive helps tasks where baseline numbers weren't run yet

**What actually happened:** the data is honest, mixed, and limited to one model. Two tasks hit infrastructure failures (ansible: stdout EOF after 17min, qutebrowser: 50min agent timeout) — those are benchflow / Daytona reliability issues, not progressive-disclosure outcomes. The three that completed all rounds:
- **openlibrary** — final reward 1.0 (regression test passed). All 3 soft-verify rounds reported 0.0, but the final hardened verifier passed. The soft-verify between rounds runs without full hardening, so its scoring can diverge from the final verifier.
- **flipt** — final 0.0 after 195 tool calls across 3 rounds. Progressive disclosure didn't unlock this task with Gemini 3.1 Pro on this run.
- **navidrome** — final 0.0 after 145 tool calls across 3 rounds. Same.

This is one model on one day, not a published comparison. The infrastructure works (round trajectories captured, soft_verify runs between rounds, BaseUser callback drives the loop). Whether progressive disclosure provides a measurable lift requires a multi-model ablation that's out of scope for this demo.

In [5]:
import json
from pathlib import Path

# Aggregated per-run results — committed alongside this notebook so the
# tables are reproducible without re-running Daytona.
results_path = Path('experiments/swebench-pro-progressive-results.json')
results = json.load(open(results_path))

print(f'{"Task":<14} {"Final":<8} {"Tools":<8} {"Time(s)":<10} {"Rounds reward":<22} {"Error"}')
print('-' * 90)
for task in ['ansible', 'flipt', 'openlibrary', 'navidrome', 'qutebrowser']:
    r = results.get(task, {})
    final = r.get('final_reward')
    final_str = f'{final}' if final is not None else '—'
    tools = r.get('tool_calls', 0)
    elapsed = r.get('elapsed_s', 0)
    rounds = r.get('rounds', [])
    rounds_str = ' / '.join(f'{x["reward"]}' for x in rounds) if rounds else '(no rounds)'
    err = (r.get('error') or '')[:30]
    print(f'{task:<14} {final_str:<8} {tools:<8} {elapsed:<10.1f} {rounds_str:<22} {err}')

passed = sum(1 for t, r in results.items() if r.get('final_reward') == 1.0)
errored = sum(1 for t, r in results.items() if r.get('error'))
print(f'\n{passed}/{len(results)} progressive runs passed final verify, {errored} hit infra errors')

Task           Final    Tools    Time(s)    Rounds reward          Error
------------------------------------------------------------------------------------------
ansible        —        0        1045.5     (no rounds)            Process closed stdout (rc=None
flipt          0.0      195      3373.8     0.0 / 0.0 / 0.0        
openlibrary    1.0      82       707.0      0.0 / 0.0 / 0.0        
navidrome      0.0      145      1514.3     0.0 / 0.0 / 0.0        
qutebrowser    —        0        3046.4     (no rounds)            Agent timed out after 3000s

1/5 progressive runs passed final verify, 2 hit infra errors


## 6. Per-task hardening opt-outs

qutebrowser ships legitimate `conftest.py` files that fix a circular import in its own source. The default verifier hardening deletes them, breaking pytest collection. The task opts out:

```toml
# .ref/swebenchpro/instance_qutebrowser__.../task.toml
[verifier.hardening]
cleanup_conftests = false
```

Available flags (all default `true` — secure-by-default):

| Flag | Effect when `false` |
|------|---------------------|
| `cleanup_conftests` | Don't delete `conftest.py` outside `/tests/` before verify |

See [`docs/progressive-disclosure.md`](../docs/progressive-disclosure.md) for the full design rationale.

## 7. Comparison with multi-agent simulated user

BenchFlow has two patterns for multi-round agent runs:

| Pattern | When to use |
|---------|-------------|
| `BaseUser` callback (this notebook) | Programmatic, rule-based, deterministic. No second LLM. Cheap. |
| Multi-role Scene with simulated-user role ([use-cases.md §1](../docs/use-cases.md#1-interactive-user-simulation-harbor-1316-equivalent)) | Open-ended, conversational. The 'user' is another LLM with full tool access. |

Both are functionally at parity with [Harbor #1316](https://github.com/harbor-ai/harbor/issues/1316), avoiding the FastMCP sidecar requirement. Choose based on whether you want a deterministic callback or another LLM agent.